In [1]:
import sys
sys.path.append("..")

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Load the FAQ documents and the search index:

In [3]:
from src.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table for the original FAQ documents:

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

### Running RAG

In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
from src.evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [7]:
rec = ground_truth[0]
rec

{'question': 'I just found this course — is it still okay to join now, or am I too late?',
 'document': '74eb249bbf'}

In [8]:
question = rec["question"]
answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join now. If you want a certificate, just make sure you submit your project while submissions are still being accepted.'

In [9]:
assistant.total_cost()

0.00059325

Get the original answer from the document ID:

In [10]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Now save both answers in one record:

In [11]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course — is it still okay to join now, or am I too late?',
 'answer_llm': 'Yes, you can still join now. If you want a certificate, just make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

### Processing all questions

In [12]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [13]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course — is it still okay to join now, or am I too late?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Before running the full batch, reset the usage we collected while testing:

In [14]:
assistant.reset_usage()

In [15]:
from concurrent.futures import ThreadPoolExecutor
from src.evaluation_utils import map_progress

Run RAG for all ground truth questions:

In [16]:
with ThreadPoolExecutor(max_workers=5) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/720 [00:00<?, ?it/s]

In [17]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [18]:
assistant.total_cost()

0.7739699999999996

In [19]:
df_answers = pd.DataFrame(answers)
print(df_answers.shape)
df_answers.head()

(720, 4)


,question,answer_llm,answer_orig,document
0,I just found this course — is it still okay to...,Yes — you can still join now. If you want a ce...,"Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still enroll if I discovered the course ...,Yes. You can still join if you discovered the ...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join the course late, can I still get a c...","Yes, but only if you finish the course with a ...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to be eligible for the ce...,"To be eligible for the certificate, you need t...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,Is it fine to take the course after it has beg...,Yes — if you can’t make the first submission w...,"Yes, but if you want to receive a certificate,...",74eb249bbf


In [20]:
df_answers.to_csv("rag-answers-new.csv", index=False)